In [1]:
import os, sys
import numpy as np

from typing import Optional, List
from frequensolve          import *
from frequensolve.seismic  import *
from frequensolve.project  import *
from frequensolve.geometry import *
from frequensolve.mesh     import *


project_path = "./output/ex01_simple/"
project = Project(path           = project_path,
                  load_if_exists = False)


sim = project.new_TD_simulation( name      = "ex01_simple",
                                 mode      = "forward",
                                 physics   = "coupled",
                                 dimension = 2,
                                 f_min     = 1.0,
                                 f_max     = 40.0,
                                 df        = 1.0 )


# ----------------------------------------------------------------------
# Define Model
# ----------------------------------------------------------------------
model = LayeredModel(dimension = 2, x_limits = [0.0, 4.0])
model.add_surface(
   name = "dune_topo",
   type = "grid",
   grid = CartesianGrid(n = [320], x0 = [0.0], x1 = [4.029]),
   file = os.path.join(project_path,"data/dunes@"),
   z_ref = 0.0,
)
                  
model.add_layer(
   name = "sand",
   properties = { "Vp" : 0.731, "Vs" : 0.3, "Rho": 1.6 }
)
                
model.add_surface(
   name = "buried_topo",
   type = "grid",
   grid = CartesianGrid(n = [1344], x0 = [0.0], dx = [3e-3]),
   file = os.path.join(project_path,"data/buried@"),
   z_ref = 0.025,
)

model.add_layer(
   name = "near_surface",
   grid = CartesianGrid(n  = [1344, 7250],
                        x0 = [0.0, 0.0],
                        dx = [1e-3, 1e-4]),
   properties = { "Vp" : os.path.join(project_path,"data/Vp_smoothed"),
                  "Vs" : os.path.join(project_path,"data/Vs_smoothed"),
                  "Rho": os.path.join(project_path,"data/Rho_smoothed")}
)
model.add_surface(z = 0.483e-1)
model.add_surface(z = 0.972e-1)
model.add_surface(z = 1.512e-1)
model.add_surface(z = 1.941e-1)
model.add_surface(z = 2.508e-1)

model.add_layer(
   name = "subsurface",
   grid = CartesianGrid(n  = [1, 7250],
                        x0 = [0.0, 0.0],
                        dx = [1e-3, 1e-4]),
   properties = { "Vp" : os.path.join(project_path,"data/Vp_1d"),
                  "Vs" : os.path.join(project_path,"data/Vs_1d"),
                  "Rho": os.path.join(project_path,"data/Rho_1d")}
)
model.add_surface(z = 3.012e-1)
model.add_surface(z = 3.3e-1)
model.add_surface(z = 3.9e-1)
model.add_surface(z = 4.791e-1)
model.add_surface(z = 5.091e-1)
model.add_surface(z = 5.412e-1)
model.add_surface(z = 5.712e-1)
model.add_surface(z = 6.03e-1)
model.add_surface(z = 6.312e-1)
model.add_surface(z = 6.99e-1)
model.add_surface(z = 7.242e-1)
model.add_surface(z = 7.551e-1)
model.add_surface(z = 7.83e-1)
model.add_surface(z = 8.16e-1)
model.add_surface(z = 9.981e-1)
model.add_surface(z = 1.0272e0)
model.add_surface(z = 1.059e0)
model.add_surface(z = 1.0941e0)
model.add_surface(z = 1.1211e0)
model.add_surface(z = 1.1532e0)
model.add_surface(z = 1.182e0)
model.add_surface(z = 1.2111e0)
model.add_surface(z = 1.2411e0)
model.add_surface(z = 1.3032e0)
model.add_surface(z = 1.335e0)
model.add_surface(z = 1.3662e0)
model.add_surface(z = 1.548e0)
model.add_surface(z = 1.581e0)
model.add_surface(z = 1.7052e0)
model.add_surface(z = 1.8201e0)
model.add_surface(z = 1.851e0)
model.add_surface(z = 1.881e0)
model.add_surface(z = 1.9692e0)
model.add_surface(z = 1.989e0)
model.add_surface(z = 2.016e0)
model.add_surface(z = 2.0451e0)
model.add_surface(z = 2.067e0)
model.add_surface(z = 2.175e0)

sim.model = model

# Define mesh generator
mesh = HexMeshGenerator(n = [16, 16], model = model)
sim.mesh = mesh

# ----------------------------------------------------------------------
# Define Boundary conditions
# ----------------------------------------------------------------------
bcm = BoundaryConditionManager(label_type = "geometric")

# Free surface BC
bc1 = BoundaryCondition(name       = "free_surface",
                        kind       = "neumann",
                        boundaries = ["z_min"])
bcm.add_BC(bc1)

# PML boundary condition
bc2 = BoundaryCondition(
   name = "pml",
   kind = "pml",
   boundaries = ["x_min", "x_max", "z_max"],
   pml_wavelengths = 2.0,
   pml_exponent    = 3.0,
   pml_constant    = 20.0
)
bcm.add_BC(bc2)

sim.boundary_conditions = bcm

# ----------------------------------------------------------------------
# Define Acquisition source, receiver geometry
# ----------------------------------------------------------------------
acq = Acquisition()

# --- Sources ---
coords = []
for x in np.linspace(0.0, 2.0, 100):
   coords.append([x, 0.0])
acq.add_source_group(kind        = "viberator",
                     frame       = "reference",
                     coordinates = coords,
                     direction   = [0.0, 1.0])

# -- Receivers ---
device = ReceiverNode(name = "geophone")
device.add_component("p"  ,"pressure")
device.add_component("u_z","displacement",[0.0, 1.0])
device.add_component("u_x","displacement",[1.0, 0.0])

coords = []
for i, x in enumerate(np.linspace(0.0, 4.0, 1000)):
   coords.append([x, 0.0])

acq.add_receiver_group(name        = "surface_geophones",
                       device      = device,
                       coordinates = coords,
                       frame       = "reference")

acq.add_receiver_group(name        = "buried_geophones",
                       device      = device,
                       coordinates = coords,
                       frame       = "physical")
sim.acquisition = acq


paraview_wrapper requires calling with pvpython (packaged with Paraview)


TypeError: Acquisition.__init__() missing 2 required positional arguments: 'samples' and 'source_group'